# TP4-AdvProgram4DS

**Read our [Gemma 3 blog](https://unsloth.ai/blog/gemma3) for what's new in Unsloth and our [Reasoning blog](https://unsloth.ai/blog/r1-reasoning) on how to train reasoning models.**

Visit our docs for all our [model uploads](https://docs.unsloth.ai/get-started/all-our-models) and [notebooks](https://docs.unsloth.ai/get-started/unsloth-notebooks).


# 0-Check System

Colab Server > GPU > T5 > 2026-10 > Name: VSL-T4-GPU

BUNHENG LOSS = 1.009 (loss validate)

In [1]:
import sys
import subprocess
from importlib import metadata

# 1. Check Python.
print("=== PYTHON ===")
print("Version:", sys.version.split()[0])
print("Location:", sys.executable)

# 2. Check the packages used by our notebook.
print("\n=== PACKAGES ===")

packages = [
    "torch", "torchvision", "unsloth", "unsloth_zoo",
    "transformers", "tokenizers", "datasets", "trl",
    "peft", "accelerate", "bitsandbytes", "triton",
    "xformers", "sentencepiece", "protobuf", "Pillow",
]

for package in packages:
    try:
        version = metadata.version(package)
        print(f"OK       {package:<18} {version}")
    except metadata.PackageNotFoundError:
        print(f"MISSING  {package}")

# 3. Check the NVIDIA GPU.
print("\n=== GPU ===")

try:
    result = subprocess.run(
        ["nvidia-smi"],
        capture_output=True,
        text=True,
    )
    print(result.stdout)
    if result.stderr:
        print(result.stderr)
except FileNotFoundError:
    print("NVIDIA GPU tool was not found. Check the selected kernel.")

# 4. Check whether installed packages have conflicting requirements.
print("\n=== PACKAGE REQUIREMENTS ===")

result = subprocess.run(
    [sys.executable, "-m", "pip", "check"],
    capture_output=True,
    text=True,
)
print(result.stdout)
if result.stderr:
    print(result.stderr)


=== PYTHON ===
Version: 3.13.16
Location: /usr/bin/python3

=== PACKAGES ===
OK       torch              2.11.0+cu130
OK       torchvision        0.26.0+cu130
OK       unsloth            2026.11.2
OK       unsloth_zoo        2026.11.2
OK       transformers       5.17.0
OK       tokenizers         0.23.3
OK       datasets           4.8.5
OK       trl                1.15.0
OK       peft               0.21.2
OK       accelerate         1.15.0
OK       bitsandbytes       0.50.2
OK       triton             3.6.0
OK       xformers           0.0.35
OK       sentencepiece      0.2.2
OK       protobuf           6.33.6
OK       Pillow             11.3.0

=== GPU ===
Wed Nov 11 07:08:48 2026       
+-----------------------------------------------------------------------------------------+
| NVIDIA-SMI 580.82.07              Driver Version: 580.82.07      CUDA Version: 13.0     |
+-----------------------------------------+------------------------+----------------------+
| GPU  Name                

In [2]:
# check "is torch GPU version"
import torch

print("PyTorch version:", torch.__version__)
print("CUDA build:", torch.version.cuda)
print("GPU available:", torch.cuda.is_available())

if torch.cuda.is_available():
    print("GPU name:", torch.cuda.get_device_name(0))

    # Do a small calculation on the GPU.
    numbers = torch.tensor([1.0, 2.0, 3.0], device="cuda")
    print("Calculation result:", numbers.sum().item())
    print("Calculation device:", numbers.device)
else:
    print("PyTorch cannot use a GPU in this session.")

PyTorch version: 2.11.0+cu130
CUDA build: 13.0
GPU available: True
GPU name: Tesla T4
Calculation result: 6.0
Calculation device: cuda:0


# 1-Installation

### Installation

In [5]:
# Install training tool
import sys
import subprocess

# Use fixed versions so our setup is easier to check.
packages = [
    "torch==2.11.0+cu130",
    "torchvision==0.26.0+cu130",
    "unsloth==2026.11.2",
    "unsloth_zoo==2026.11.2",
    "transformers==5.17.0",
    "trl==1.15.0",
    "bitsandbytes==0.50.2",
    "huggingface-hub<2",
    "requests==2.32.4",
    "Pillow==11.3.0",
    "jedi",
]

# Download xformers built for our PyTorch and CUDA setup.
xformers_wheel = (
    "https://download.pytorch.org/whl/cu130/"
    "xformers-0.0.35-py39-none-manylinux_2_28_x86_64.whl"
)
packages.append(xformers_wheel)

# Install into the Python that runs this notebook.
subprocess.check_call(
    [sys.executable, "-m", "pip", "install", *packages]
)

print("Installation finished.")
print("To comfirm please ReRun the system check !")

Installation finished.


In [ ]:
# New Installation Replaced

# %%capture
# import os
# if "COLAB_" not in "".join(os.environ.keys()):
#     !pip install unsloth vllm
# else:
#     # [NOTE] Do the below ONLY in Colab! Use [[pip install unsloth vllm]]
#     !pip install --no-deps unsloth vllm
# # Install latest Hugging Face for Gemma-3!
# !pip install --no-deps git+https://github.com/huggingface/transformers@v4.49.0-Gemma-3
# !pip install bitsandbytes
# !pip install unsloth_zoo
# !pip install msgspec
# !pip install blake3
# !pip install gguf
# !pip install networkx==2.8.8 --force-reinstall

In [ ]:
# New Installation Replaced

# #@title Colab Extra Install { display-mode: "form" }
# %%capture
# import os
# if "COLAB_" not in "".join(os.environ.keys()):
#     !pip install unsloth vllm
# else:
#     !pip install --no-deps unsloth vllm
#     # [NOTE] Do the below ONLY in Colab! Use [[pip install unsloth vllm]]
#     # Skip restarting message in Colab
#     import sys, re, requests; modules = list(sys.modules.keys())
#     for x in modules: sys.modules.pop(x) if "PIL" in x or "google" in x else None
#     !pip install --no-deps bitsandbytes accelerate xformers==0.0.29.post3 peft trl triton cut_cross_entropy unsloth_zoo
#     !pip install sentencepiece protobuf datasets huggingface_hub hf_transfer

#     # vLLM requirements - vLLM breaks Colab due to reinstalling numpy
#     f = requests.get("https://raw.githubusercontent.com/vllm-project/vllm/refs/heads/main/requirements/common.txt").content
#     with open("vllm_requirements.txt", "wb") as file:
#         file.write(re.sub(rb"(transformers|numpy|xformers)[^\n]{1,}\n", b"", f))
#     !pip install -r vllm_requirements.txt

# 2.Unsloth

`FastModel` supports loading nearly any model now! This includes Vision and Text models!

In [ ]:
from unsloth import FastModel # fast model is for load and preppare the model
import torch

fourbit_models = [
    # 4bit dynamic quants for superior accuracy and low memory use
    "unsloth/gemma-3-1b-it-unsloth-bnb-4bit",
    "unsloth/gemma-3-4b-it-unsloth-bnb-4bit",
    "unsloth/gemma-3-12b-it-unsloth-bnb-4bit",
    "unsloth/gemma-3-27b-it-unsloth-bnb-4bit",

    # Other popular models!
    "unsloth/Llama-3.1-8B",
    "unsloth/Llama-3.2-3B",
    "unsloth/Llama-3.3-70B",
    "unsloth/mistral-7b-instruct-v0.3",
    "unsloth/Phi-4",
] # More models at https://huggingface.co/unsloth

model, tokenizer = FastModel.from_pretrained(
    model_name = "unsloth/gemma-3-4b-it",
    max_seq_length = 2048, # Choose any for long context!
    load_in_4bit = True,  # 4 bit quantization to reduce memory
    load_in_8bit = False, # [NEW!] A bit more accurate, uses 2x memory
    full_finetuning = False, # [NEW!] We have full finetuning now!
    # token = "hf_...", # use one if using gated models
)

🦥 Unsloth: Will patch your computer to enable 2x faster free finetuning.
🦥 Unsloth Zoo will now patch everything to make training faster!
==((====))==  Unsloth 2026.11.2: Fast Gemma3 patching. Transformers: 5.17.0.
   \\   /|    Tesla T4. Num GPUs = 1. Max memory: 14.563 GB. Platform: Linux.
O^O/ \_/ \    Torch: 2.11.0+cu130. CUDA: 7.5. CUDA Toolkit: 13.0. Triton: 3.6.0
\        /    Bfloat16 = FALSE. FA [Xformers = 0.0.35. FA2 = False]
 "-____-"     Free license: http://github.com/unslothai/unsloth
Unsloth: Fast downloading is enabled - ignore downloading bars which are red colored!
Unsloth: Using float16 precision for gemma3 won't work! Using float32.


Loading weights:   0%|          | 0/883 [00:00<?, ?it/s]

Skipping model.vision_tower.encoder.layers.0.self_attn.k_proj: no quant_state found
Skipping model.vision_tower.encoder.layers.0.self_attn.v_proj: no quant_state found
Skipping model.vision_tower.encoder.layers.0.self_attn.q_proj: no quant_state found
Skipping model.vision_tower.encoder.layers.0.self_attn.out_proj: no quant_state found
Skipping model.vision_tower.encoder.layers.0.mlp.fc1: no quant_state found
Skipping model.vision_tower.encoder.layers.0.mlp.fc2: no quant_state found
Skipping model.vision_tower.encoder.layers.1.self_attn.k_proj: no quant_state found
Skipping model.vision_tower.encoder.layers.1.self_attn.v_proj: no quant_state found
Skipping model.vision_tower.encoder.layers.1.self_attn.q_proj: no quant_state found
Skipping model.vision_tower.encoder.layers.1.self_attn.out_proj: no quant_state found
Skipping model.vision_tower.encoder.layers.1.mlp.fc1: no quant_state found
Skipping model.vision_tower.encoder.layers.1.mlp.fc2: no quant_state found
Skipping model.vision_to

We now add LoRA adapters so we only need to update a small amount of parameters!

In [ ]:
model = FastModel.get_peft_model(
    model,
    finetune_vision_layers     = False, # Turn off for just text!
    finetune_language_layers   = True,  # Should leave on!
    finetune_attention_modules = True,  # Attention good for GRPO, Include attention parts (parts that connect information across tokens)
    finetune_mlp_modules       = True,  # SHould leave on always!

    r = 8,           # Larger = higher accuracy, but might overfit
    lora_alpha = 8,  # Recommended alpha == r at least
    lora_dropout = 0,
    bias = "none",
    random_state = 3407,
)

# 3.Data Prep
<a name="Data"></a>
### Data Prep
We now use the `Gemma-3` format for conversation style finetunes. We use [Maxime Labonne's FineTome-100k](https://huggingface.co/datasets/mlabonne/FineTome-100k) dataset in ShareGPT style. Gemma-3 renders multi turn conversations like below:

```
<bos><start_of_turn>user
Hello!<end_of_turn>
<start_of_turn>model
Hey there!<end_of_turn>
```

We use our `get_chat_template` function to get the correct chat template. We support `zephyr, chatml, mistral, llama, alpaca, vicuna, vicuna_old, phi3, llama3, phi4, qwen2.5, gemma3` and more.

In [10]:
from unsloth.chat_templates import get_chat_template
tokenizer = get_chat_template(
    tokenizer,
    chat_template = "gemma-3",
)
print("Gemma 3 message format is ready.")

Gemma 3 message format is ready.


## A. Import DataSet

In [ ]:
# from datasets import load_dataset
# dataset = load_dataset('csv',data_files="dataasilida_cleaned.csv")

Generating train split: 0 examples [00:00, ? examples/s]

In [16]:
# existing code loads data; it does not fix these problems. this code expects separate input and output columns, so it would fail.
# The checking cell is just an extra step to help see the problem. After clean the CSV, we can use the existing load_dataset code (above, cell10).

import csv
from pathlib import Path
from collections import Counter

csv_path = Path("/content/dataasilida_cleaned.csv")

if not csv_path.exists():
    raise FileNotFoundError("Upload the CSV to Colab first.")

with csv_path.open(encoding="utf-8-sig", newline="") as file:
    reader = csv.reader(file)
    header = next(reader, [])
    raw_rows = list(reader)

wrong_columns = []
missing_inputs = []
missing_outputs = []
valid_pairs = []

for record_number, row in enumerate(raw_rows, start=2):
    # Each row should contain a question and an answer.
    if len(row) != 2:
        wrong_columns.append(record_number)
        continue

    question = row[0].strip()
    answer = row[1].strip()

    if not question:
        missing_inputs.append(record_number)

    if not answer:
        missing_outputs.append(record_number)

    valid_pairs.append((question, answer))

# Count repeated question-and-answer pairs.
pair_counts = Counter(valid_pairs)
duplicate_count = sum(count - 1 for count in pair_counts.values())

print("=== COLUMNS ===")
print("Expected:", ["input", "output"])
print("Found:", header)
print("Correct header:", header == ["input", "output"])

print("\n=== ROWS ===")
print("Total data rows:", len(raw_rows))
print("Rows without exactly two fields:", len(wrong_columns))
print("Affected records:", wrong_columns)

print("\n=== MISSING VALUES ===")
print("Missing questions:", missing_inputs)
print("Missing answers:", missing_outputs)
print("These checks cover only rows with two fields.")

print("\n=== DUPLICATES ===")
print("Extra copies of repeated pairs:", duplicate_count)

=== COLUMNS ===
Expected: ['input', 'output']
Found: ['input,output']
Correct header: False

=== ROWS ===
Total data rows: 521
Rows without exactly two fields: 7
Affected records: [98, 99, 101, 393, 395, 419, 424]

=== MISSING VALUES ===
Missing questions: []
Missing answers: []
These checks cover only rows with two fields.

=== DUPLICATES ===
Extra copies of repeated pairs: 2


## B.Clean dataset

In [ ]:
clean_rows = []
fixed_count = 0

for record_number, row in enumerate(raw_rows, start=2):
    if len(row) == 1:
        text = row[0]

        if record_number == 419:
            # This question ends at the first question mark.
            question, separator, answer = text.partition("?")

            if not separator:
                raise ValueError("Cannot split record 419.")

            row = [question + "?", answer]
        else:
            # Split the question and answer at the first comma.
            row = text.split(",", 1)

        fixed_count += 1

    # Check that both fields exist and contain text.
    if len(row) != 2:
        raise ValueError(f"Check record {record_number}.")

    question = row[0].strip()
    answer = row[1].strip()

    if not question or not answer:
        raise ValueError(f"Empty field in record {record_number}.")

    clean_rows.append([question, answer])

# Save a new file with two separate column names.
clean_path = csv_path.with_name("dataasilida_ready.csv")

with clean_path.open(
    "w", encoding="utf-8", newline=""
) as file:
    writer = csv.writer(file)
    writer.writerow(["input", "output"])
    writer.writerows(clean_rows)

print("Fixed rows:", fixed_count)
print("Saved examples:", len(clean_rows))
print("New file:", clean_path)

In [15]:
dataset['train']['input'][0]

ValueError: Column 'input' doesn't exist.

In [13]:
def formatting_prompts_func(examples):
    instructions = examples["input"]
    outputs      = examples["output"]
    texts = []
    for instruction, output in zip(instructions, outputs):
      messages = [
        {
            "role": "user", 'content': instruction.strip(" ")
          },
          {
            "role": "assistant", 'content': output
          }
      ]
      #tokenizer.apply_chat_template(messages, add_generation_prompt=False , tokenize= False)
      text = tokenizer.apply_chat_template(messages, add_generation_prompt=False , tokenize= False)
      texts.append(text)
    return {"text":texts}



In [14]:
dataset = load_dataset('csv', data_files="data_kh.csv")
dataset = dataset.map(formatting_prompts_func, batched = True,)


In [15]:
dataset['train']['text'][0]

"<bos><start_of_turn>user\nយើងទៅមុខ១២.៨៧ម៉ែត្រល្ប១.៨៨ម៉ែត្រ\\វិ<end_of_turn>\n<start_of_turn>model\n{'action':'move','params':{'linear_speed':1.88,'distance':12.87,'is_forward':True,'unit':'meter'}}<end_of_turn>\n"

Let's see how row 100 looks like!

In [10]:
dataset['train']['input'][0]

'យើងទៅមុខ១២.៨៧ម៉ែត្រល្ប១.៨៨ម៉ែត្រ\\វិ'

In [ ]:
dataset['train']['input'][0]

Let's see how the chat template did! Notice `Gemma-3` default adds a `<bos>`!

In [12]:
dataset['train']['output'][00]

"{'action':'move','params':{'linear_speed':1.88,'distance':12.87,'is_forward':True,'unit':'meter'}}"

# 4-Train the Model
<a name="Train"></a>
### Train the model
Now let's use Huggingface TRL's `SFTTrainer`! More docs here: [TRL SFT docs](https://huggingface.co/docs/trl/sft_trainer). We do 60 steps to speed things up, but you can set `num_train_epochs=1` for a full run, and turn off `max_steps=None`.

In [49]:
# from trl import SFTTrainer, SFTConfig
# trainer = SFTTrainer(
#     model = model,
#     tokenizer = tokenizer,
#     train_dataset = dataset,
#     eval_dataset = None, # Can set up evaluation!
#     args = SFTConfig(
#         dataset_text_field = "text",
#         per_device_train_batch_size = 2,
#         gradient_accumulation_steps = 4, # Use GA to mimic batch size!
#         warmup_steps = 5,
#         # num_train_epochs = 1, # Set this for 1 full training run.
#         max_steps = 60,
#         learning_rate = 2e-4, # Reduce to 2e-5 for long training runs
#         logging_steps = 1,
#         optim = "adamw_8bit",
#         weight_decay = 0.01,
#         lr_scheduler_type = "linear",
#         seed = 3407,
#         report_to = "none", # Use this for WandB etc
#     ),
# )

Unsloth: Switching to float32 training since model cannot work with float16


KeyError: "Invalid key: 0. Please first select a split. For example: `my_dataset_dictionary['train'][0]`. Available splits: ['train']"

In [16]:
from trl import SFTTrainer
from transformers import TrainingArguments
from unsloth import is_bfloat16_supported

trainer = SFTTrainer(
    model = model,
    tokenizer = tokenizer,
    train_dataset = dataset['train'],
    dataset_text_field = "text",
    #max_seq_length = max_seq_length,
    dataset_num_proc = 2,
    packing = False, # Can make training 5x faster for short sequences.
    args = TrainingArguments(
        per_device_train_batch_size = 2,
        gradient_accumulation_steps = 4,
        warmup_steps = 5,
        max_steps = 60,
        learning_rate = 2e-4,
        fp16 = not is_bfloat16_supported(),
        bf16 = is_bfloat16_supported(),
        logging_steps = 1,
        optim = "adamw_8bit",
        weight_decay = 0.01,
        lr_scheduler_type = "linear",
        seed = 3407,
        output_dir = "outputs",
        report_to = "none", # Use this for WandB etc
    ),
)

Unsloth: Switching to float32 training since model cannot work with float16


Unsloth: Tokenizing ["text"] (num_proc=2):   0%|          | 0/2930 [00:00<?, ? examples/s]

We also use Unsloth's `train_on_completions` method to only train on the assistant outputs and ignore the loss on the user's inputs. This helps increase accuracy of finetunes!

In [53]:
from unsloth.chat_templates import train_on_responses_only
trainer = train_on_responses_only(
    trainer,
    instruction_part = "<start_of_turn>user\n",
    response_part = "<start_of_turn>model\n",
)

Map (num_proc=2):   0%|          | 0/2930 [00:00<?, ? examples/s]

Let's verify masking the instruction part is done! Let's print the 100th row again:

In [54]:
tokenizer.decode(trainer.train_dataset[100]["input_ids"])

"<bos><bos><start_of_turn>user\nបង្វែឆ្វេង១៦៩ដឺក្រេក្នុងល្បឿន១ម៉ែត្រ\\វិនាទី<end_of_turn>\n<start_of_turn>model\n{'action':'rotate','params':{'angular_velocity':1,'angle':169,'is_clockwise':False,'unit':'degrees'}}<end_of_turn>\n"

Now let's print the masked out example - you should see only the answer is present:

In [55]:
tokenizer.decode([tokenizer.pad_token_id if x == -100 else x for x in trainer.train_dataset[100]["labels"]]).replace(tokenizer.pad_token, " ")

"                                     {'action':'rotate','params':{'angular_velocity':1,'angle':169,'is_clockwise':False,'unit':'degrees'}}<end_of_turn>\n"

In [ ]:
# @title Show current memory stats
gpu_stats = torch.cuda.get_device_properties(0)
start_gpu_memory = round(torch.cuda.max_memory_reserved() / 1024 / 1024 / 1024, 3)
max_memory = round(gpu_stats.total_memory / 1024 / 1024 / 1024, 3)
print(f"GPU = {gpu_stats.name}. Max memory = {max_memory} GB.")
print(f"{start_gpu_memory} GB of memory reserved.")

GPU = Tesla T4. Max memory = 14.741 GB.
5.457 GB of memory reserved.


Let's train the model! To resume a training run, set `trainer.train(resume_from_checkpoint = True)`

In [17]:
trainer_stats = trainer.train()

==((====))==  Unsloth - 2x faster free finetuning | Num GPUs used = 1
   \\   /|    Num examples = 2,930 | Num Epochs = 1 | Total steps = 60
O^O/ \_/ \    Batch size per device = 2 | Gradient accumulation steps = 4
\        /    Data Parallel GPUs = 1 | Total batch size (2 x 4 x 1) = 8
 "-____-"     Trainable parameters = 14,901,248/4,000,000,000 (0.37% trained)
`use_cache=True` is incompatible with gradient checkpointing. Setting `use_cache=False`.


Step,Training Loss
1,5.379100
2,6.010000
3,6.118500
4,4.986100
5,4.926300
6,4.668400
7,3.962200
8,3.925600
9,3.060200
10,2.892200


In [ ]:
# @title Show final memory and time stats
used_memory = round(torch.cuda.max_memory_reserved() / 1024 / 1024 / 1024, 3)
used_memory_for_lora = round(used_memory - start_gpu_memory, 3)
used_percentage = round(used_memory / max_memory * 100, 3)
lora_percentage = round(used_memory_for_lora / max_memory * 100, 3)
print(f"{trainer_stats.metrics['train_runtime']} seconds used for training.")
print(
    f"{round(trainer_stats.metrics['train_runtime']/60, 2)} minutes used for training."
)
print(f"Peak reserved memory = {used_memory} GB.")
print(f"Peak reserved memory for training = {used_memory_for_lora} GB.")
print(f"Peak reserved memory % of max memory = {used_percentage} %.")
print(f"Peak reserved memory for training % of max memory = {lora_percentage} %.")

470.5985 seconds used for training.
7.84 minutes used for training.
Peak reserved memory = 6.309 GB.
Peak reserved memory for training = 0.852 GB.
Peak reserved memory % of max memory = 42.799 %.
Peak reserved memory for training % of max memory = 5.78 %.


# 5-Inference
<a name="Inference"></a>
### Inference
Let's run the model via Unsloth native inference! According to the `Gemma-3` team, the recommended settings for inference are `temperature = 1.0, top_p = 0.95, top_k = 64`

In [21]:
from unsloth.chat_templates import get_chat_template
tokenizer = get_chat_template(
    tokenizer,
    chat_template = "gemma-3",
)
messages = [{
    "role": "user",
    "content": [{
        "type" : "text",
        "text" : "ទៅមុខ១០មហើយបត់ឆ្វេង",
    }]
}]
text = tokenizer.apply_chat_template(
    messages,
    add_generation_prompt = True, # Must add for generation
)
outputs = model.generate(
    **tokenizer([text], return_tensors = "pt").to("cuda"),
    max_new_tokens = 64, # Increase for longer outputs!
    # Recommended Gemma-3 settings!
    temperature = 1.0, top_p = 0.95, top_k = 64,
)
tokenizer.batch_decode(outputs)

["<bos><start_of_turn>user\nទៅមុខ១០មហើយបត់ឆ្វេង<end_of_turn>\n<start_of_turn>model\n{'action':'move','params':{'distance':10,'is_forward':True,'unit':'meter'}}\\n{'action':'rotate','params':{'angle':90,'is_clockwise':False,'unit':'degree'}}<end_of_turn>"]

 You can also use a `TextStreamer` for continuous inference - so you can see the generation token by token, instead of waiting the whole time!

In [20]:
messages = [{
    "role": "user",
    "content": [{"type" : "text", "text" : "ទៅមុខ១០ម",}]
}]
text = tokenizer.apply_chat_template(
    messages,
    add_generation_prompt = True, # Must add for generation
)

from transformers import TextStreamer
_ = model.generate(
    **tokenizer([text], return_tensors = "pt").to("cuda"),
    max_new_tokens = 64, # Increase for longer outputs!
    # Recommended Gemma-3 settings!
    temperature = 1.0, top_p = 0.95, top_k = 64,
    streamer = TextStreamer(tokenizer, skip_prompt = True),
)

{'action':'move','params':{'distance':10,'is_forward':True,'unit':'meter'}}<end_of_turn>


# 6. Saving, loading fineturned models
<a name="Save"></a>
### Saving, loading finetuned models
To save the final model as LoRA adapters, either use Huggingface's `push_to_hub` for an online save or `save_pretrained` for a local save.

**[NOTE]** This ONLY saves the LoRA adapters, and not the full model. To save to 16bit or GGUF, scroll down!

In [22]:
model.save_pretrained("gemma-3")  # Local saving
tokenizer.save_pretrained("gemma-3")
# model.push_to_hub("HF_ACCOUNT/gemma-3", token = "...") # Online saving
# tokenizer.push_to_hub("HF_ACCOUNT/gemma-3", token = "...") # Online saving

['gemma-3/processor_config.json']

Now if you want to load the LoRA adapters we just saved for inference, set `False` to `True`:

In [32]:
if False:
    from unsloth import FastModel
    model, tokenizer = FastModel.from_pretrained(
        model_name = "lora_model", # YOUR MODEL YOU USED FOR TRAINING
        max_seq_length = 2048,
        load_in_4bit = True,
    )

messages = [{
    "role": "user",
    "content": [{"type" : "text", "text" : "បន្តទៅក្រោយ៩ម",}]
}]
text = tokenizer.apply_chat_template(
    messages,
    add_generation_prompt = True, # Must add for generation
)

from transformers import TextStreamer
_ = model.generate(
    **tokenizer([text], return_tensors = "pt").to("cuda"),
    max_new_tokens = 64, # Increase for longer outputs!
    # Recommended Gemma-3 settings!
    temperature = 1.0, top_p = 0.95, top_k = 64,
    streamer = TextStreamer(tokenizer, skip_prompt = True),
)

{'action':'move','params':{'distance':9,'is_forward':False,'unit':'meter'}}<end_of_turn>


# 7.Saving to float16 for VLLM

We also support saving to `float16` directly for deployment! We save it in the folder `gemma-3-finetune`. Set `if False` to `if True` to let it run!

In [ ]:
if False: # Change to True to save finetune!
    model.save_pretrained_merged("gemma-3-finetune", tokenizer)

If you want to upload / push to your Hugging Face account, set `if False` to `if True` and add your Hugging Face token and upload location!

In [ ]:
if False: # Change to True to upload finetune
    model.push_to_hub_merged(
        "HF_ACCOUNT/gemma-3-finetune", tokenizer,
        token = "hf_..."
    )

In [ ]:
if False: # Change to True to save to GGUF
    model.save_pretrained_gguf(
        "gemma-3-finetune",
        quantization_type = "Q8_0", # For now only Q8_0, BF16, F16 supported
    )

Likewise, if you want to instead push to GGUF to your Hugging Face account, set `if False` to `if True` and add your Hugging Face token and upload location!

In [ ]:
if False: # Change to True to upload GGUF
    model.push_to_hub_gguf(
        "gemma-3-finetune",
        quantization_type = "Q8_0", # Only Q8_0, BF16, F16 supported
        repo_id = "HF_ACCOUNT/gemma-finetune-gguf",
        token = "hf_...",
    )